<a href="https://colab.research.google.com/github/christophermagno/christophermagno/blob/main/Personal%20Health%20Analysis/health_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🩻 Health Data Extraction

Using [**Garmin Connect API**](https://github.com/cyberjunky/python-garminconnect)

From Garmin watch

In [ ]:
# TODO: Gather solar data with timestamp ranges per day. Will need to store in a separate dataset and link through `Date`.

## ⬇️ Install Garmin python package

In [ ]:
%pip install garminconnect

In [ ]:
import re
import os
import sys
import logging
import importlib
import datetime
import requests
from getpass import getpass
from pathlib import Path
from tqdm import tqdm

import pandas as pd
import numpy as np

from google.colab import drive

from garth.exc import GarthException, GarthHTTPError
from garminconnect import (
    Garmin,
    GarminConnectAuthenticationError,
    GarminConnectConnectionError,
    GarminConnectTooManyRequestsError,
)

# Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s', force=True)
log = logging.getLogger('Personal Health Analysis')

# Google Drive
drive.mount('/content/drive')
path_health_data = Path('/content/drive/MyDrive/Colab Notebooks/Data Analyst/christophermagno/Projects/Personal Health Analysis/personal_health_data_2026.csv')
path_activities_data = Path('/content/drive/MyDrive/Colab Notebooks/Data Analyst/christophermagno/Projects/Personal Health Analysis/personal_acitivites_data_2026.csv')

# Date format that Garmin likes
DATE_FORMAT = '%Y-%m-%d'

# Update the dataset even if the csv file exists
# Use caution because average heart rate data is gathered from granular
# minute-to-minute data which is only stored up to a certain date
FORCE_UPDATE = False

Mounted at /content/drive


## ⌚️ Garmin helper functions to interact with API
Taken from example file provided in documentation to ensure safe usage with API

In [ ]:
def safe_api_call(api_method, *args, **kwargs):
    """
    Safe API call wrapper with comprehensive error handling.

    This demonstrates the error handling patterns used throughout the library.
    Returns (success: bool, result: Any, error_message: str)
    """
    try:
        result = api_method(*args, **kwargs)
        return True, result, None

    except GarthHTTPError as e:
        # Handle specific HTTP errors gracefully
        error_str = str(e)
        status_code = getattr(getattr(e, "response", None), "status_code", None)

        if status_code == 400 or "400" in error_str:
            return (
                False,
                None,
                "Endpoint not available (400 Bad Request) - Feature may not be enabled for your account",
            )
        elif status_code == 401 or "401" in error_str:
            return (
                False,
                None,
                "Authentication required (401 Unauthorized) - Please re-authenticate",
            )
        elif status_code == 403 or "403" in error_str:
            return (
                False,
                None,
                "Access denied (403 Forbidden) - Account may not have permission",
            )
        elif status_code == 404 or "404" in error_str:
            return (
                False,
                None,
                "Endpoint not found (404) - Feature may have been moved or removed",
            )
        elif status_code == 429 or "429" in error_str:
            return (
                False,
                None,
                "Rate limit exceeded (429) - Please wait before making more requests",
            )
        elif status_code == 500 or "500" in error_str:
            return (
                False,
                None,
                "Server error (500) - Garmin's servers are experiencing issues",
            )
        elif status_code == 503 or "503" in error_str:
            return (
                False,
                None,
                "Service unavailable (503) - Garmin's servers are temporarily unavailable",
            )
        else:
            return False, None, f"HTTP error: {e}"

    except FileNotFoundError:
        return (
            False,
            None,
            "No valid tokens found. Please login with your email/password to create new tokens.",
        )

    except GarminConnectAuthenticationError as e:
        return False, None, f"Authentication issue: {e}"

    except GarminConnectConnectionError as e:
        return False, None, f"Connection issue: {e}"

    except GarminConnectTooManyRequestsError as e:
        return False, None, f"Rate limit exceeded: {e}"

    except Exception as e:
        return False, None, f"Unexpected error: {e}"

def get_credentials():
    """Get email and password from environment or user input."""
    email = os.getenv("EMAIL")
    password = os.getenv("PASSWORD")

    if not email:
        email = input("Login email: ")
    if not password:
        password = getpass("Enter password: ")

    return email, password

def init_api() -> Garmin | None:
    """Initialize Garmin API with authentication and token management."""

    # Configure token storage
    tokenstore = os.getenv("GARMINTOKENS", "~/.garminconnect")
    tokenstore_path = Path(tokenstore).expanduser()

    print(f"🔐 Token storage: {tokenstore_path}")

    # Check if token files exist
    if tokenstore_path.exists():
        print("📄 Found existing token directory")
        token_files = list(tokenstore_path.glob("*.json"))
        if token_files:
            print(
                f"🔑 Found {len(token_files)} token file(s): {[f.name for f in token_files]}"
            )
        else:
            print("⚠️ Token directory exists but no token files found")
    else:
        print("📭 No existing token directory found")

    # First try to login with stored tokens
    try:
        print("🔄 Attempting to use saved authentication tokens...")
        garmin = Garmin()
        garmin.login(str(tokenstore_path))
        print("✅ Successfully logged in using saved tokens!")
        return garmin

    except (
        FileNotFoundError,
        GarthHTTPError,
        GarminConnectAuthenticationError,
        GarminConnectConnectionError,
    ):
        print("🔑 No valid tokens found. Requesting fresh login credentials.")

    # Loop for credential entry with retry on auth failure
    while True:
        try:
            # Get credentials
            email, password = get_credentials()

            print("� Logging in with credentials...")
            garmin = Garmin(
                email=email, password=password, is_cn=False, return_on_mfa=True
            )
            result1, result2 = garmin.login()

            if result1 == "needs_mfa":
                print("🔐 Multi-factor authentication required")

                mfa_code = input("Please enter your MFA code: ")
                print("🔄 Submitting MFA code...")

                try:
                    garmin.resume_login(result2, mfa_code)
                    print("✅ MFA authentication successful!")

                except GarthHTTPError as garth_error:
                    # Handle specific HTTP errors from MFA
                    error_str = str(garth_error)
                    if "429" in error_str and "Too Many Requests" in error_str:
                        print("❌ Too many MFA attempts")
                        print("💡 Please wait 30 minutes before trying again")
                        sys.exit(1)
                    elif "401" in error_str or "403" in error_str:
                        print("❌ Invalid MFA code")
                        print("💡 Please verify your MFA code and try again")
                        continue
                    else:
                        # Other HTTP errors - don't retry
                        print(f"❌ MFA authentication failed: {garth_error}")
                        sys.exit(1)

                except GarthException as garth_error:
                    print(f"❌ MFA authentication failed: {garth_error}")
                    print("💡 Please verify your MFA code and try again")
                    continue

            # Save tokens for future use
            garmin.garth.dump(str(tokenstore_path))
            print(f"💾 Authentication tokens saved to: {tokenstore_path}")
            print("✅ Login successful!")
            return garmin

        except GarminConnectAuthenticationError:
            print("❌ Authentication failed:")
            print("💡 Please check your username and password and try again")
            # Continue the loop to retry
            continue

        except (
            FileNotFoundError,
            GarthHTTPError,
            GarminConnectConnectionError,
            requests.exceptions.HTTPError,
        ) as err:
            print(f"❌ Connection error: {err}")
            print("💡 Please check your internet connection and try again")
            return None

        except KeyboardInterrupt:
            print("\n👋 Cancelled by user")
            return None


### Get Garmin client

In [ ]:
api = init_api()

🔐 Token storage: /root/.garminconnect
📄 Found existing token directory
🔑 Found 2 token file(s): ['oauth2_token.json', 'oauth1_token.json']
🔄 Attempting to use saved authentication tokens...
✅ Successfully logged in using saved tokens!


## Helper functions for datetime

In [ ]:
def _get_date_string(date):
    return date.strftime(DATE_FORMAT)

def today():
    return datetime.date.today().strftime(DATE_FORMAT)

def convert_epoch_to_datetime(epoch):
    try:
        return datetime.datetime.fromtimestamp(epoch / 1000)
    except TypeError:
        return pd.NaT

def get_date_range(start=None, rng=None):
    start = start or datetime.datetime.today()
    rng = rng or int(start.strftime('%j'))
    dates = reversed([start - datetime.timedelta(days=x) for x in range(rng)])
    return [x.strftime(DATE_FORMAT) for x in dates]


## Helper functions to gather and organize Garmin data

Some useful methods from the Garmin class to use
* get_stats - using
* get_heart_rates - using
* get_sleep_data - using
* get_fitnessage_data - using
* get_activities - using
*
* get_steps_data
* get_daily_steps
* get_floors
* get_stress_data
* get_rhr_day
* get_hrv_data

Others to look at
* get_activities_fordate
* get_earned_badges

In [ ]:
def get_device():
    """
    deviceId
    imageUrl
    productDisplayName
    deviceTypeSimpleName
    """
    devices = safe_api_call(api.get_devices)[1]
    return devices[0]

def get_sleep_data(date):

    sleep_data = {}

    to_pop = [
        'id',
        'userProfilePK',
        'napTimeSeconds',
        'sleepWindowConfirmed',
        'sleepWindowConfirmationType',
        'autoSleepStartTimestampGMT',
        'autoSleepEndTimestampGMT',
        'sleepQualityTypePK',
        'sleepResultTypePK',
        'deviceRemCapable',
        'retro',
        'sleepFromDevice',
        'sleepScores',
        'sleepScoreInsight',
        'sleepScorePersonalizedInsight',
        'sleepVersion',
        'averageSPO2',
        'averageSpO2HR',
        'lowestSPO2'
    ]

    data = safe_api_call(api.get_sleep_data, date)[1]

    sleep_data.update(data['dailySleepDTO'])
    if 'sleepScores' in sleep_data:
        sleep_data['sleepScore'] = sleep_data['sleepScores']['overall']['value']
        sleep_data['sleepScoreQuality'] = sleep_data['sleepScores']['overall']['qualifierKey']
        sleep_data['stressSleepQuality'] = sleep_data['sleepScores']['stress']['qualifierKey']
        sleep_data['awakeCountQuality'] = sleep_data['sleepScores']['awakeCount']['qualifierKey']
        sleep_data['remSleepQuality'] = sleep_data['sleepScores']['remPercentage']['qualifierKey']
        sleep_data['restlessnessSleepQuality'] = sleep_data['sleepScores']['restlessness']['qualifierKey']
        sleep_data['lightSleepQuality'] = sleep_data['sleepScores']['lightPercentage']['qualifierKey']
        sleep_data['deepSleepQuality'] = sleep_data['sleepScores']['deepPercentage']['qualifierKey']
    else:
        sleep_data['sleepScore'] = None
        sleep_data['sleepScoreQuality'] = None
        sleep_data['stressSleepQuality'] = None
        sleep_data['awakeCountQuality'] = None
        sleep_data['remSleepQuality'] = None
        sleep_data['restlessnessSleepQuality'] = None
        sleep_data['lightSleepQuality'] = None
        sleep_data['deepSleepQuality'] = None

    # result['sleepHeartRate'] = data['sleepHeartRate']
    sleep_data['avgOvernightHrv'] = data.get('avgOvernightHrv')
    sleep_data['hrvStatus'] = data.get('hrvStatus')
    sleep_data['restingHeartRate'] = data.get('restingHeartRate')

    # Convert timestamp to datetime
    for item in ['sleepStartTimestampGMT', 'sleepEndTimestampGMT', 'sleepStartTimestampLocal', 'sleepEndTimestampLocal']:
        sleep_data[item] = convert_epoch_to_datetime(sleep_data[item])

    for key in to_pop:
        try:
            sleep_data.pop(key)
        except KeyError as e:
            pass

    return sleep_data

def get_hydration_data(date):
    data = safe_api_call(api.get_hydration_data, date)[1]
    hydration_data = {
        'hydrationValueInML': data['valueInML'],
        'hydrationGoalInML': data['goalInML'],
        'sweatLossInML': data['sweatLossInML']
    }
    return hydration_data

def get_solar_data():
    """
    Example output
    {
    'localConnectDate': '2025-12-28',
     'userProfilePk': 126748136,
     'deviceId': 3476417250,
     'solarInputReadings': [{'readingTimestampLocal': '2025-12-28T00:00:00.0',
       'readingTimestampGmt': '2025-12-28T08:00:00.0',
       'solarUtilization': 0.0,
       'notChargingTooHot': False,
       'notChargingTooCold': False,
       'notChargingBatteryFull': False,
       'notChargingExternalPower': False,
       'notChargingUserDisabled': False,
       'notChargingOther': True,
       'activityTimeGainMs': 0,
       'charging': False,
       'interpolated': None},
    """

    d = safe_api_call(api.get_device_solar_data, get_device()['deviceId'], dates[-3], dates[-1])[1]['solarDailyDataDTOs']
    print(d)

### Get Activities Data

In [ ]:
def get_activities_data(dates):
    """
    - ownerFullName
    - ownerProfileImageUrlMedium

    activityId
    activityName
    activityType: {
        typeId
        typeKey
    }
    locationName (hiking?)
    startTimeLocal
    startTimeGMT
    endTimeGMT
    beginTimestamp

    distance
    duration
    elapsedDuration
    movingDuration
    - elevationGain (hiking)
    - elevationLoss (hiking)
    averageSpeed
    maxSpeed
    hasPolyline
    hasImages
    ownerId
    ownerFullName
    ownerProfileImageUrlMedium
    calories
    bmrCalories
    averageHR
    maxHR
    steps
    aerobicTrainingEffect
    anaerobicTrainingEffect
    summarizedExerciseSets: [
        category
        reps
        volume
        duration
        sets
        maxWeight
    ]

    lapCount
    totalSets
    activeSets
    totalReps

    activityTrainingLoad
    minActivityLapDuration

    moderateIntensityMinutes
    vigorousIntensityMinutes

    hrTimeInZone_1
    hrTimeInZone_2
    hrTimeInZone_3
    hrTimeInZone_4
    hrTimeInZone_5

    pr
    """

    result = []

    keys_to_get = [
        'activityId',
        'activityName',
        # activityType: {
        #     typeId
        #     typeKey
        # }

        # 'ownerId',
        # 'ownerFullName',
        # 'ownerProfileImageUrlMedium',
        # 'hasImages',

        # 'locationName',  # (hiking?)
        # 'elevationGain',  # (hiking)
        # 'elevationLoss',  # (hiking)

        'startTimeLocal',
        'startTimeGMT',
        'endTimeGMT',
        'beginTimestamp',

        'pr',

        'duration',
        'elapsedDuration',
        'movingDuration',

        'calories',
        'bmrCalories',

        'steps',
        'distance',

        'averageSpeed',
        'maxSpeed',

        'averageHR',
        'maxHR',
        'hrTimeInZone_1',
        'hrTimeInZone_2',
        'hrTimeInZone_3',
        'hrTimeInZone_4',
        'hrTimeInZone_5',

        'lapCount',
        'totalSets',
        'activeSets',
        'totalReps',

        'aerobicTrainingEffect',
        'anaerobicTrainingEffect',
        'moderateIntensityMinutes',
        'vigorousIntensityMinutes',
        'activityTrainingLoad'
    ]
    activities = safe_api_call(api.get_activities_by_date, dates[0], dates[-1])[1]
    for activity in activities:
        activity_data = {}
        for key in keys_to_get:
            activity_data[key] = activity.get(key, None)
            if key == 'activityName':
                activity_data['activityType'] = activity['activityType'].get('typeKey', 'Unknown').title().replace('_', ' ')

            # Get total calories
            if key == 'bmrCalories':
                activity_data['totalCalories'] = activity['bmrCalories'] + activity['calories']

        result.append(activity_data)
    return result

### Get Health Data

In [ ]:
def get_health_data(date):
    """
    """

    to_pop = [
        'userProfileId',
        'userDailySummaryId',
        'burnedKilocalories',
        'wellnessActiveKilocalories',
        'netRemainingKilocalories',
        'rule',
        'wellnessStartTimeGmt',
        'wellnessStartTimeLocal',
        'wellnessEndTimeGmt',
        'wellnessEndTimeLocal',
        'durationInMilliseconds',
        'wellnessDescription',
        'includesWellnessData',
        'includesActivityData',
        'includesCalorieConsumedData',
        'privacyProtected',
        'floorsAscended',
        'floorsDescended',
        'lastSevenDaysAvgRestingHeartRate',
        'source',
        'lastSyncTimestampGMT',
        'bodyBatteryMostRecentValue',
        'bodyBatteryVersion',
        # 'averageSpo2',
        'lowestSpO2Value',
        'highestSpO2Value',
        'lowestSpo2',
        'latestSpo2',
        'latestSpo2ReadingTimeGmt',
        'latestSpo2ReadingTimeLocal',
        'latestSpo2ReadingTimeLocalaverageMonitoringEnvironmentAltitude',
        'restingCaloriesFromActivity',
        'latestRespirationValue',
        'latestRespirationTimeGMT',
        'respirationAlgorithmVersion',
        'ageGroup',
        'averageMonitoringEnvironmentAltitude',
        # 'bodyBatteryChargedValue',
        # 'bodyBatteryDrainedValue',
        # 'bodyBatteryHighestValue',
        # 'bodyBatteryLowestValue',
        # 'bodyBatteryDuringSleep',
        'wellnessKilocalories',
        'consumedKilocalories',
        'remainingKilocalories',
        'netCalorieGoal',
        'wellnessDistanceMeters',
        'userNote',
        'sleepingSeconds',
        'minAvgHeartRate',
        'maxAvgHeartRate',
        'abnormalHeartRateAlertsCount',
        'unmeasurableSleepSeconds',
        # 'measurableAsleepDuration',
        # 'measurableAwakeDuration',
        'stressPercentage',
        'restStressPercentage',
        'activityStressPercentage',
        'uncategorizedStressPercentage',
        'lowStressPercentage',
        'mediumStressPercentage',
        'highStressPercentage',
        'restStressDuration',
        'userFloorsAscendedGoal',
    ]

    success, health_data, _ = safe_api_call(api.get_stats, date)
    success, result, _ = safe_api_call(api.get_fitnessage_data, date)
    if success:
        health_data['fitnessAge'] = int(result['fitnessAge'])
    else:
        health_data['fitnessAge'] = None

    # Get heart rate data

    # Heart rate data
    hdata = safe_api_call(api.get_heart_rates, date)[1]['heartRateValues']
    if hdata:
        heart_rates = [v[1] for v in hdata if v[1] is not None]
        health_data['avgHeartRate'] = float(np.array(heart_rates).mean())

    # Get sleep data
    health_data.update(get_sleep_data(date))

    # Get hydration data
    health_data.update(get_hydration_data(date))

    for key in to_pop:
        try:
            health_data.pop(key)
        except KeyError as e:
            pass

    # Convert/Add some columns
    convert_dict = {}
    for key, value in health_data.items():
        if value:
            if 'Meters' in key:
                convert_dict[key.replace('Meters', 'Miles')] = value / 1609
            elif 'Seconds' in key:
                convert_dict[key.replace('Seconds', 'Hours')] = value / 3600
            elif 'Duration' in key:
                convert_dict[key.replace('Duration', 'Hours')] = value / 3600
            elif 'Minutes' in key:
                convert_dict[key.replace('Minutes', 'Hours')] = value / 60
            elif 'InML' in key:
                convert_dict[key.replace('InML', 'InCups')] = value / 240

    # Update
    health_data.update(convert_dict)

    return health_data

2026-01-18 08:02:51,675 - INFO - True


{'displayName': 'f5b57009-0b6f-4dc7-a5c5-c162d279aa0f',
 'totalKilocalories': None,
 'activeKilocalories': None,
 'bmrKilocalories': None,
 'totalSteps': None,
 'totalDistanceMeters': None,
 'calendarDate': '2026-01-18',
 'uuid': None,
 'dailyStepGoal': None,
 'highlyActiveSeconds': None,
 'activeSeconds': None,
 'sedentarySeconds': None,
 'moderateIntensityMinutes': None,
 'vigorousIntensityMinutes': None,
 'floorsAscendedInMeters': None,
 'floorsDescendedInMeters': None,
 'intensityMinutesGoal': None,
 'minHeartRate': None,
 'maxHeartRate': None,
 'restingHeartRate': None,
 'averageStressLevel': None,
 'maxStressLevel': None,
 'stressDuration': None,
 'activityStressDuration': None,
 'uncategorizedStressDuration': None,
 'totalStressDuration': None,
 'lowStressDuration': None,
 'mediumStressDuration': None,
 'highStressDuration': None,
 'stressQualifier': None,
 'measurableAwakeDuration': None,
 'measurableAsleepDuration': None,
 'bodyBatteryChargedValue': None,
 'bodyBatteryDrainedV

In [ ]:
def get_healths_data(dates=None):
    data = []
    for date in tqdm(dates or get_date_range()):
        data.append(get_health_data(date))
    return data

## 📆 Get Date Range

In [ ]:
# current_date = datetime.datetime.strptime('12-31-2025', '%m-%d-%Y')
current_date = datetime.datetime.today()
dates = get_date_range(current_date)
log.info(f'Number of dates: {len(dates)}')

2026-01-18 07:55:21,782 - INFO - Number of dates: 18


## ⚕️ Build Health Data
### Create the Health Dataframe

Remapping dictionary to fix the columns names and order

In [ ]:
remap_health_columns = {
    'uuid': 'uuid',
    'calendarDate': 'Date',

    'fitnessAge': 'Fitness Age',

    # Calories
    'totalKilocalories': 'Calories',
    'activeKilocalories': 'Active Calories',
    'bmrKilocalories': 'Resting Calories',

    # Hydration
    'hydrationValueInML': 'Hydration Value In ML',
    'hydrationGoalInML': 'Hydration Goal In ML',
    'sweatLossInML': 'Sweat Loss In ML',
    'hydrationValueInCups': 'Hydration Value In Cups',
    'hydrationGoalInCups': 'Hydration Goal In Cups',
    'sweatLossInCups': 'Sweat Loss In Cups',

    # Heart Rate
    'avgHeartRate': 'Average Heart Rate',
    'minHeartRate': 'Min Heart Rate',
    'maxHeartRate': 'Max Heart Rate',
    'restingHeartRate': 'Resting Heart Rate',
    'hrvStatus': 'Heart Rate Variability Qualifier',

    # Peripheral Oxyge Saturation
    'averageSpo2': 'Average Sp 02',

    # Respiration
    'avgWakingRespirationValue': 'Avg Waking Respiration Value',
    'highestRespirationValue': 'Highest Respiration Value',
    'lowestRespirationValue': 'Lowest Respiration Value',

    # Steps/Distance
    'totalSteps': 'Total Steps',
    'totalDistanceMeters': 'Total Distance Meters',
    'totalDistanceMiles': 'Total Distance Miles',
    'dailyStepGoal': 'Daily Step Goal',

    # Floors
    'floorsAscendedInMeters': 'Floors Ascended In Meters',
    'floorsDescendedInMeters': 'Floors Descended In Meters',

    'floorsAscendedInMiles': 'Floors Ascended In Miles',
    'floorsDescendedInMiles': 'Floors Descended In Miles',

    # Activity
    'activeSeconds': 'Active Seconds',
    'highlyActiveSeconds': 'Highly Active Seconds',
    'sedentarySeconds': 'Sedentary Seconds',
    'moderateIntensityMinutes': 'Moderate Intensity Minutes',
    'vigorousIntensityMinutes': 'Vigorous Intensity Minutes',
    'intensityMinutesGoal': 'Intensity Minutes Goal',

    'activeHours': 'Active Hours',
    'highlyActiveHours': 'Highly Active Hours',
    'sedentaryHours': 'Sedentary Hours',
    'moderateIntensityHours': 'Moderate Intensity Hours',
    'vigorousIntensityHours': 'Vigorous Intensity Hours',
    'intensityHoursGoal': 'Intensity Hours Goal',

    # Stress
    'averageStressLevel': 'Average Stress Level',
    'totalStressDuration': 'Total Stress Seconds',
    'stressDuration': 'Stress Seconds',
    'maxStressLevel': 'Max Stress Level',
    'uncategorizedStressDuration': 'Uncategorized Stress Seconds',
    'lowStressDuration': 'Low Stress Seconds',
    'mediumStressDuration': 'Medium Stress Seconds',
    'highStressDuration': 'High Stress Seconds',
    'activityStressDuration': 'Activity Stress Seconds',
    'stressQualifier': 'Stress Qualifier',

    'stressHours': 'Stress Hours',
    'activityStressHours': 'Activity Stress Hours',
    'uncategorizedStressHours': 'Uncategorized Stress Hours',
    'totalStressHours': 'Total Stress Hours',
    'lowStressHours': 'Low Stress Hours',
    'mediumStressHours': 'Medium Stress Hours',
    'highStressHours': 'High Stress Hours',

    # Body battery
    'bodyBatteryAtWakeTime': 'Body Battery',
    'bodyBatteryChargedValue': 'Body Battery Charged Value',
    'bodyBatteryDrainedValue': 'Body Battery Drained Value',
    'bodyBatteryHighestValue': 'Body Battery Highest Value',
    'bodyBatteryLowestValue': 'Body Battery Lowest Value',
    'bodyBatteryDuringSleep': 'Body Battery During Sleep',

    # Sleep data
    'sleepStartTimestampGMT': 'Sleep Start Timestamp GMT',
    'sleepEndTimestampGMT': 'Sleep End Timestamp GMT',
    'sleepStartTimestampLocal': 'Sleep Start Timestamp Local',
    'sleepEndTimestampLocal': 'Sleep End Timestamp Local',

    'sleepTimeSeconds': 'Sleep Time Seconds',
    'sleepScore': 'Sleep Score',
    'sleepScoreQuality': 'Sleep Quality',
    'sleepScoreFeedback': 'Sleep Feedback',

    'measurableAsleepDuration': 'Measurable Asleep Seconds',
    'measurableAwakeDuration': 'Measurable Awake Seconds',

    'lightSleepSeconds': 'Light Sleep Seconds',
    'deepSleepSeconds': 'Deep Sleep Seconds',
    'remSleepSeconds': 'Rem Sleep Seconds',
    'awakeSleepSeconds': 'Awake Sleep Seconds',

    'sleepTimeHours': 'Sleep Time Hours',
    'measurableAsleepHours': 'Measurable Asleep Hours',
    'measurableAwakeHours': 'Measurable Awake Hours',
    'deepSleepHours': 'Deep Sleep Hours',
    'lightSleepHours': 'Light Sleep Hours',
    'remSleepHours': 'Rem Sleep Hours',
    'awakeSleepHours': 'Awake Sleep Hours',

    'averageRespirationValue': 'Average Respiration Value',
    'awakeCount': 'Awake Count',
    'avgSleepStress': 'Avg Sleep Stress',

    'stressSleepQuality': 'Stress Sleep Quality',
    'awakeCountQuality': 'Awake Count Quality',
    'remSleepQuality': 'Rem Sleep Quality',
    'restlessnessSleepQuality': 'Restlessness Sleep Quality',
    'lightSleepQuality': 'Light Sleep Quality',
    'deepSleepQuality': 'Deep Sleep Quality',

    'avgOvernightHrv': 'Average Overnight Hrv',
}

### Generate the Health Data

In [ ]:
# Build the health data set
# If data already exists, just read the data and generate health data for
# new days
if path_health_data.exists() and not FORCE_UPDATE:
    df_health = pd.read_csv(path_health_data)
    try:
        df_health = df_health.drop('Unnamed: 0', axis=1)  # I don't know why it keeps adding this column...
    except:
        pass
    dates_to_update = sorted(list(set(dates).difference(set(df_health['Date'].tolist()))))
    log.info(f'Updating Dates: {dates_to_update}')
    health_data = get_healths_data(dates_to_update)

    df_health = pd.concat([df_health, pd.DataFrame(health_data).convert_dtypes()])
else:
    # Otherwise generate the data YTD
    health_data = get_healths_data(dates)
    df_health = pd.DataFrame(health_data).convert_dtypes()
df_health.tail()

2026-01-18 08:03:59,204 - INFO - Updating Dates: ['2026-01-18']
100%|██████████| 1/1 [00:00<00:00,  2.34it/s]

                            displayName totalKilocalories activeKilocalories  \
0  f5b57009-0b6f-4dc7-a5c5-c162d279aa0f              None               None   

  bmrKilocalories totalSteps totalDistanceMeters calendarDate  uuid  \
0            None       None                None   2026-01-18  None   

  dailyStepGoal highlyActiveSeconds  ... remSleepQuality  \
0          None                None  ...            None   

  restlessnessSleepQuality lightSleepQuality deepSleepQuality avgOvernightHrv  \
0                     None              None             None            None   

  hrvStatus hydrationValueInML hydrationGoalInML sweatLossInML  \
0      None               None          2839.056          None   

  hydrationGoalInCups  
0             11.8294  

[1 rows x 63 columns]


,uuid,Date,Fitness Age,Calories,Active Calories,Resting Calories,Hydration Value In ML,Hydration Goal In ML,Sweat Loss In ML,Hydration Value In Cups,...,remSleepQuality,restlessnessSleepQuality,lightSleepQuality,deepSleepQuality,avgOvernightHrv,hrvStatus,hydrationValueInML,hydrationGoalInML,sweatLossInML,hydrationGoalInCups
13,8ef852657bcb48179a8f3b89f363b5da,2026-01-14,32.0,1666.0,155.0,1511.0,2129.292,2860.056,21.0,8.872050,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>
14,ce8f28f6fb604937aac8a9264986dfaf,2026-01-15,32.0,2862.0,1351.0,1511.0,2839.056,2839.056,NaN,11.829400,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>
15,6646f2bca23347b8a8fbc7f684515ea9,2026-01-16,32.0,3069.0,1558.0,1511.0,3075.644,4381.056,1542.0,12.815183,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>
16,b00f2774b1ca44e0aeb1e0dcf3096270,2026-01-17,32.0,1122.0,278.0,844.0,473.176,2865.056,26.0,1.971567,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>
0,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,None,None,None,None,None,None,None,2839.056,None,11.8294


### Update the Health Dataset
Read current data and just append rows of new health data

In [ ]:
df_health1 = df_health.rename(remap_health_columns, axis=1)[remap_health_columns.values()]

In [ ]:
title_columns = ['Qualifier', 'Quality', 'Feedback']
for col, series in df_health1.items():
    for item in title_columns:
        if re.search(item, col, re.IGNORECASE):
            df_health1[col] = series.str.title().str.replace('_', ' ')
df_health1.head()

,uuid,Date,Fitness Age,Calories,Active Calories,Resting Calories,Hydration Value In ML,Hydration Goal In ML,Sweat Loss In ML,Hydration Value In Cups,...,Average Respiration Value,Awake Count,Avg Sleep Stress,Stress Sleep Quality,Awake Count Quality,Rem Sleep Quality,Restlessness Sleep Quality,Light Sleep Quality,Deep Sleep Quality,Average Overnight Hrv
0,9c0ea781368e44a48875fb994e4a861e,2026-01-01,30,3123,1612,1511,2839.056,2839.056,<NA>,11.8294,...,15,2,22,Fair,Fair,Good,Good,Excellent,Fair,49
1,016cffd3cf1f4317a2a8094154e6c766,2026-01-02,31,3523,2012,1511,2839.056,2839.056,<NA>,11.8294,...,14,2,34,Poor,Fair,Poor,Fair,Fair,Good,36
2,dbcd3a5aadba42239bb2384a18c405c4,2026-01-03,31,1919,408,1511,1656.116,2839.056,<NA>,6.900483,...,14,1,23,Fair,Good,Fair,Good,Fair,Fair,46
3,f1e59b524d5b416a8a0621285053595b,2026-01-04,31,3236,1725,1511,2839.056,2839.056,<NA>,11.8294,...,14,0,11,Excellent,Excellent,Fair,Excellent,Good,Poor,57
4,a1c79b2e5b9f4957af82b41fb1bb6f86,2026-01-05,31,2901,1390,1511,2839.056,2890.056,51,11.8294,...,14,0,31,Poor,Excellent,Good,Excellent,Good,Good,36


### Convert some types

In [ ]:
df_health1['Date'] = pd.to_datetime(df_health1['Date'])
for col in ['Sleep Start Timestamp GMT', 'Sleep End Timestamp GMT', 'Sleep Start Timestamp Local', 'Sleep End Timestamp Local']:
    df_health1[col] = pd.to_datetime(df_health1[col])

### Export the data

In [ ]:
df_health1.to_csv(path_health_data)

## 🏋🏽‍♂️ Create the Activities Dataframe

In [ ]:
remap_activities_columns = {
    'activityId': 'id',
    'activityName': 'Activity Name',
    'activityType': 'Activity Type',

    'startTimeLocal': 'Start Time',
    'endTimeLocal': 'End Time',

    'pr': 'Personal Record',

    'duration': 'Duration',

    'calories': 'Active Calories',
    'bmrCalories': 'Resting Calories',
    'totalCalories': 'Total Calories',

    'steps': 'Steps',
    'distance': 'Distance',

    'averageSpeed': 'Average Speed',
    'maxSpeed': 'Max Speed',

    'averageHR': 'Average Heart Rate',
    'maxHR': 'Max Heart Rate',
    'hrTimeInZone_1': 'Heart Rate Zone 1 Duration',
    'hrTimeInZone_2': 'Heart Rate Zone 2 Duration',
    'hrTimeInZone_3': 'Heart Rate Zone 3 Duration',
    'hrTimeInZone_4': 'Heart Rate Zone 4 Duration',
    'hrTimeInZone_5': 'Heart Rate Zone 5 Duration',

    'lapCount': 'Lap Count',
    'totalSets': 'Total Sets',
    'activeSets': 'Active Sets',
    'totalReps': 'Total Reps',

    'aerobicTrainingEffect': 'Aerobic Training Effect',
    'anaerobicTrainingEffect': 'Anaerobic Training Effect',
    'moderateIntensityMinutes': 'Moderate Intensity Minutes',
    'vigorousIntensityMinutes': 'Vigorous Intensity Minutes',
    'activityTrainingLoad': 'Activity Training Load',
}

In [ ]:
activities_data = get_activities_data(dates)
df_activities = pd.DataFrame(activities_data).convert_dtypes()
df_activities['startTimeLocal'] = pd.to_datetime(df_activities['startTimeLocal'])

### Get end time from startTimeLocal and duration

In [ ]:
df_activities['endTimeLocal'] = df_activities.apply(lambda row: row['startTimeLocal'] + datetime.timedelta(seconds=row['duration']), axis=1)
df_activities['endTimeLocal'] = pd.to_datetime(df_activities['endTimeLocal'])

### Rename and reorder the columns

In [ ]:
df_activities1 = df_activities.rename(columns=remap_activities_columns)[remap_activities_columns.values()]

### Fill some null values

In [ ]:
df_activities1[['Distance', 'Average Speed', 'Max Speed']] = df_activities1[['Distance', 'Average Speed', 'Max Speed']].fillna(0)

### Export the data

In [ ]:
df_activities1.to_csv(path_activities_data)